In [ ]:
%sql

SELECT
    fila_excel,
    columna_excel,
    coordenada_excel,
    valor_texto,
    tipo_valor
FROM catalog_au.bronze.sfc_indicadores_raw
WHERE archivo_origen = 'ig_2024_01.xlsx'
  AND fila_excel BETWEEN 1 AND 12
ORDER BY
    fila_excel,
    columna_excel;

In [ ]:
%sql

SELECT
    fila_excel,
    MAX(
        CASE WHEN columna_excel = 1
        THEN valor_texto END
    ) AS columna_A,
    MAX(
        CASE WHEN columna_excel = 2
        THEN valor_texto END
    ) AS columna_B,
    MAX(
        CASE WHEN columna_excel = 3
        THEN valor_texto END
    ) AS columna_C,
    MAX(
        CASE WHEN columna_excel = 4
        THEN valor_texto END
    ) AS columna_D,
    MAX(
        CASE WHEN columna_excel = 10
        THEN valor_texto END
    ) AS columna_J
FROM catalog_au.bronze.sfc_indicadores_raw
WHERE archivo_origen = 'ig_2024_01.xlsx'
  AND fila_excel BETWEEN 13 AND 45
GROUP BY fila_excel
ORDER BY fila_excel;

In [ ]:
%sql

SELECT
    columna_excel,
    MAX(
        CASE WHEN fila_excel = 1
        THEN valor_texto END
    ) AS identificador,
    MAX(
        CASE WHEN fila_excel = 11
        THEN valor_texto END
    ) AS nombre_entidad,
    MAX(
        CASE WHEN fila_excel = 12
        THEN valor_texto END
    ) AS periodo
FROM catalog_au.bronze.sfc_indicadores_raw
WHERE archivo_origen = 'ig_2024_01.xlsx'
  AND fila_excel IN (1, 11, 12)
  AND columna_excel BETWEEN 4 AND 20
GROUP BY columna_excel
ORDER BY columna_excel;

In [ ]:
%sql

SELECT
    columna_excel,

    MAX(
        CASE
            WHEN fila_excel = 1
            THEN valor_texto
        END
    ) AS codigo_entidad,

    TRIM(
        MAX(
            CASE
                WHEN fila_excel = 11
                THEN valor_texto
            END
        )
    ) AS nombre_entidad,

    MAX(
        CASE
            WHEN fila_excel = 12
            THEN valor_texto
        END
    ) AS periodo

FROM catalog_au.bronze.sfc_indicadores_raw

WHERE archivo_origen = 'ig_2024_01.xlsx'
  AND fila_excel IN (1, 11, 12)
  AND columna_excel >= 21

GROUP BY columna_excel

ORDER BY columna_excel;

In [ ]:
%sql

WITH encabezados AS (
    SELECT
        archivo_origen,
        anio_origen,
        columna_excel,

        MAX(
            CASE WHEN fila_excel = 1
            THEN valor_texto END
        ) AS codigo_entidad,

        TRIM(
            MAX(
                CASE WHEN fila_excel = 11
                THEN valor_texto END
            )
        ) AS nombre_entidad

    FROM catalog_au.bronze.sfc_indicadores_raw

    WHERE fila_excel IN (1, 11)
      AND columna_excel >= 4

    GROUP BY
        archivo_origen,
        anio_origen,
        columna_excel
)

SELECT
    anio_origen,
    COUNT(*) AS total_columnas_entidades,

    SUM(
        CASE
            WHEN codigo_entidad RLIKE '^1-'
            AND nombre_entidad IS NOT NULL
            THEN 1 ELSE 0
        END
    ) AS bancos_individuales,

    SUM(
        CASE
            WHEN codigo_entidad RLIKE '^2-'
            THEN 1 ELSE 0
        END
    ) AS corporaciones_financieras,

    SUM(
        CASE
            WHEN codigo_entidad RLIKE '^4-'
            THEN 1 ELSE 0
        END
    ) AS companias_financiamiento,

    SUM(
        CASE
            WHEN codigo_entidad RLIKE '^32-'
            THEN 1 ELSE 0
        END
    ) AS cooperativas_financieras

FROM encabezados

GROUP BY anio_origen
ORDER BY anio_origen;

In [ ]:
from pyspark.sql import functions as F

catalogo = "catalog_au"

tabla_bronze = f"{catalogo}.bronze.sfc_indicadores_raw"
tabla_silver = f"{catalogo}.silver.sfc_indicadores_v2"

print("Origen:", tabla_bronze)
print("Destino:", tabla_silver)

In [ ]:
df_bronze = spark.table(tabla_bronze)

df_encabezados = (
    df_bronze
    .filter(
        (F.col("fila_excel").isin(1, 11, 12)) &
        (F.col("columna_excel") >= 4)
    )
    .groupBy(
        "archivo_origen",
        "anio_origen",
        "columna_excel"
    )
    .agg(
        F.max(
            F.when(
                F.col("fila_excel") == 1,
                F.col("valor_texto")
            )
        ).alias("codigo_entidad"),

        F.max(
            F.when(
                F.col("fila_excel") == 11,
                F.trim(F.col("valor_texto"))
            )
        ).alias("nombre_entidad"),

        F.max(
            F.when(
                F.col("fila_excel") == 12,
                F.col("valor_texto")
            )
        ).alias("periodo")
    )
    .filter(
        F.col("codigo_entidad").isNotNull() &
        F.col("nombre_entidad").isNotNull()
    )
)

display(
    df_encabezados.orderBy(
        "anio_origen",
        "columna_excel"
    )
)

In [ ]:
display(
    df_encabezados
    .groupBy("anio_origen")
    .agg(
        F.count("*").alias("total_entidades"),
        F.countDistinct("codigo_entidad").alias("codigos_unicos"),
        F.countDistinct("nombre_entidad").alias("nombres_unicos")
    )
    .orderBy("anio_origen")
)

In [ ]:
df_conceptos = (
    df_bronze
    .filter(
        (F.col("columna_excel").between(1, 3)) &
        (F.col("fila_excel") >= 13) &
        F.col("valor_texto").isNotNull()
    )
    .select(
        "archivo_origen",
        "anio_origen",
        "fila_excel",
        "columna_excel",
        F.trim(F.col("valor_texto")).alias("nombre_concepto")
    )
    .withColumn(
        "nivel_jerarquia",
        F.col("columna_excel")
    )
    .filter(F.col("nombre_concepto") != "")
)

display(
    df_conceptos.orderBy(
        "anio_origen",
        "fila_excel"
    )
)

In [ ]:
display(
    df_conceptos
    .groupBy("anio_origen")
    .agg(
        F.count("*").alias("total_conceptos"),
        F.countDistinct("nombre_concepto").alias("conceptos_unicos"),
        F.sum(
            F.when(F.col("nivel_jerarquia") == 1, 1)
             .otherwise(0)
        ).alias("nivel_1"),
        F.sum(
            F.when(F.col("nivel_jerarquia") == 2, 1)
             .otherwise(0)
        ).alias("nivel_2"),
        F.sum(
            F.when(F.col("nivel_jerarquia") == 3, 1)
             .otherwise(0)
        ).alias("nivel_3")
    )
    .orderBy("anio_origen")
)

In [ ]:
df_valores = (
    df_bronze
    .filter(
        (F.col("fila_excel") >= 13) &
        (F.col("columna_excel") >= 4) &
        F.col("valor_texto").isNotNull()
    )
    .select(
        "archivo_origen",
        "anio_origen",
        "fila_excel",
        "columna_excel",
        "valor_texto",
        "tipo_valor",
        "fecha_ingesta"
    )
)

display(
    df_valores
    .groupBy("anio_origen")
    .agg(
        F.count("*").alias("total_valores"),
        F.countDistinct("fila_excel").alias("filas_con_valores"),
        F.countDistinct("columna_excel").alias("columnas_con_valores")
    )
    .orderBy("anio_origen")
)

In [ ]:
df_indicadores_base = (
    df_valores.alias("v")

    .join(
        df_encabezados.alias("e"),
        (
            (F.col("v.archivo_origen") == F.col("e.archivo_origen")) &
            (F.col("v.anio_origen") == F.col("e.anio_origen")) &
            (F.col("v.columna_excel") == F.col("e.columna_excel"))
        ),
        "inner"
    )

    .join(
        df_conceptos.alias("c"),
        (
            (F.col("v.archivo_origen") == F.col("c.archivo_origen")) &
            (F.col("v.anio_origen") == F.col("c.anio_origen")) &
            (F.col("v.fila_excel") == F.col("c.fila_excel"))
        ),
        "inner"
    )

    .select(
        F.col("v.archivo_origen").alias("archivo_origen"),
        F.col("v.anio_origen").alias("anio_origen"),
        F.col("v.fila_excel").alias("fila_excel"),
        F.col("v.columna_excel").alias("columna_excel"),

        F.col("e.codigo_entidad").alias("codigo_entidad"),
        F.col("e.nombre_entidad").alias("nombre_entidad"),
        F.col("e.periodo").alias("periodo"),

        F.col("c.nombre_concepto").alias("nombre_concepto"),
        F.col("c.nivel_jerarquia").alias("nivel_jerarquia"),

        F.col("v.valor_texto").alias("valor_texto"),
        F.col("v.tipo_valor").alias("tipo_valor"),
        F.col("v.fecha_ingesta").alias("fecha_ingesta")
    )
)

display(
    df_indicadores_base
    .orderBy(
        "anio_origen",
        "fila_excel",
        "columna_excel"
    )
    .limit(30)
)

In [ ]:
display(
    df_indicadores_base
    .groupBy("anio_origen")
    .agg(
        F.count("*").alias("total_indicadores"),
        F.countDistinct("codigo_entidad").alias("entidades"),
        F.countDistinct("fila_excel").alias("conceptos"),
        F.countDistinct("periodo").alias("periodos")
    )
    .orderBy("anio_origen")
)

In [ ]:
from pyspark.sql import functions as F

df_analisis_valores = (
    df_indicadores_base
    .withColumn(
        "valor_limpio",
        F.trim(F.col("valor_texto"))
    )
    .withColumn(
        "clasificacion_valor",
        F.when(
            F.col("valor_limpio").rlike(
                r"^[+-]?\d+(\.\d+)?$"
            ),
            "NUMERICO"
        )
        .when(
            F.col("valor_limpio").rlike(
                r"^[+-]?\d{1,3}(,\d{3})+(\.\d+)?$"
            ),
            "NUMERICO_CON_SEPARADOR"
        )
        .when(
            F.col("valor_limpio").rlike(
                r"^[+-]?\d+(\.\d+)?%$"
            ),
            "PORCENTAJE"
        )
        .when(
            F.col("valor_limpio").isin(
                "-", "--", "N/A", "NA", ""
            ),
            "SIN_DATO"
        )
        .otherwise("OTRO_FORMATO")
    )
)

display(
    df_analisis_valores
    .groupBy(
        "anio_origen",
        "clasificacion_valor"
    )
    .agg(
        F.count("*").alias("total_registros")
    )
    .orderBy(
        "anio_origen",
        "clasificacion_valor"
    )
)

In [ ]:
display(
    df_analisis_valores
    .filter(
        F.col("clasificacion_valor") != "NUMERICO"
    )
    .select(
        "anio_origen",
        "nombre_entidad",
        "nombre_concepto",
        "valor_texto",
        "clasificacion_valor"
    )
    .orderBy(
        "anio_origen",
        "clasificacion_valor"
    )
    .limit(50)
)

In [ ]:
from pyspark.sql import functions as F

df_resumen_calidad = (
    df_analisis_valores
    .groupBy("anio_origen")
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("clasificacion_valor") == "NUMERICO", 1
            ).otherwise(0)
        ).alias("numericos"),

        F.sum(
            F.when(
                F.col("clasificacion_valor") == "NUMERICO_CON_SEPARADOR", 1
            ).otherwise(0)
        ).alias("numericos_con_separador"),

        F.sum(
            F.when(
                F.col("clasificacion_valor") == "PORCENTAJE", 1
            ).otherwise(0)
        ).alias("porcentajes"),

        F.sum(
            F.when(
                F.col("clasificacion_valor") == "SIN_DATO", 1
            ).otherwise(0)
        ).alias("sin_dato"),

        F.sum(
            F.when(
                F.col("clasificacion_valor") == "OTRO_FORMATO", 1
            ).otherwise(0)
        ).alias("otros_formatos"),

        F.sum(
            F.when(
                F.col("valor_limpio") == "#DIV/0!", 1
            ).otherwise(0)
        ).alias("errores_division")
    )
    .orderBy("anio_origen")
)

display(df_resumen_calidad)

In [ ]:
from pyspark.sql import functions as F

df_formatos_pendientes = (
    df_analisis_valores
    .filter(
        (F.col("clasificacion_valor") == "OTRO_FORMATO") &
        (F.col("valor_limpio") != "#DIV/0!")
    )
)

display(
    df_formatos_pendientes
    .groupBy("valor_limpio", "tipo_valor")
    .agg(
        F.count("*").alias("total_registros")
    )
    .orderBy(F.desc("total_registros"))
)

In [ ]:
from pyspark.sql import functions as F

patron_numerico = (
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?$"
)

df_normalizado = (
    df_analisis_valores

    .withColumn(
        "es_numerico",
        F.col("valor_limpio").rlike(patron_numerico)
    )

    .withColumn(
        "valor_decimal",
        F.when(
            F.col("es_numerico"),
            F.expr(
                "try_cast(valor_limpio AS DECIMAL(38,18))"
            )
        )
    )

    .withColumn(
        "valor_indicador",
        F.expr(
            "try_cast(valor_decimal AS DECIMAL(38,18))"
        )
    )

    .withColumn(
        "estado_validacion",
        F.when(
            F.col("valor_limpio") == "#DIV/0!",
            "ERROR_ORIGEN"
        )
        .when(
            F.col("clasificacion_valor") == "SIN_DATO",
            "SIN_DATO"
        )
        .when(
            F.col("es_numerico") &
            F.col("valor_decimal").isNull(),
            "ERROR_CONVERSION"
        )
        .when(
            F.col("es_numerico") &
            F.col("valor_indicador").isNull(),
            "ERROR_PRECISION"
        )
        .when(
            F.col("es_numerico") &
            (F.col("valor_decimal") != 0) &
            (F.col("valor_indicador") == 0),
            "PERDIDA_PRECISION"
        )
        .when(
            F.col("es_numerico"),
            "VALIDO"
        )
        .otherwise("FORMATO_NO_RECONOCIDO")
    )
)

In [ ]:
display(
    df_normalizado
    .groupBy(
        "anio_origen",
        "estado_validacion"
    )
    .agg(
        F.count("*").alias("total_registros")
    )
    .orderBy(
        "anio_origen",
        "estado_validacion"
    )
)

In [ ]:
from pyspark.sql import functions as F

patron_numerico = (
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?$"
)

df_precision = (
    df_analisis_valores
    .filter(F.col("valor_limpio").rlike(patron_numerico))
    .withColumn(
        "valor_38_18",
        F.expr("try_cast(valor_limpio AS DECIMAL(38,18))")
    )
    .withColumn(
        "estado_precision",
        F.when(
            F.col("valor_38_18").isNull(),
            "ERROR_CONVERSION"
        )
        .when(
            (F.col("valor_38_18") == 0) &
            (F.expr("try_cast(valor_limpio AS DOUBLE)") != 0),
            "PERDIDA_PRECISION"
        )
        .otherwise("CONVERSION_POSIBLE")
    )
)

display(
    df_precision
    .groupBy("anio_origen", "estado_precision")
    .count()
    .orderBy("anio_origen", "estado_precision")
)

In [ ]:
from pyspark.sql import functions as F

# Meses en español
meses = {
    "ene": 1, "feb": 2, "mar": 3,
    "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9,
    "oct": 10, "nov": 11, "dic": 12
}

mapa_meses = F.create_map(
    *[
        elemento
        for mes, numero in meses.items()
        for elemento in (F.lit(mes), F.lit(numero))
    ]
)

df_silver_preparado = (
    df_normalizado

    # Extraer el mes desde el encabezado
    .withColumn(
        "mes_texto",
        F.lower(
            F.substring(F.trim(F.col("periodo")), 1, 3)
        )
    )

    .withColumn(
        "mes",
        mapa_meses[F.col("mes_texto")].cast("int")
    )

    # Construir fecha de corte: último día del mes
    .withColumn(
        "fecha_corte",
        F.last_day(
            F.make_date(
                F.col("anio_origen"),
                F.col("mes"),
                F.lit(1)
            )
        )
    )

    .withColumn(
        "anio",
        F.year("fecha_corte")
    )

    # Código técnico basado en la fila del reporte
    .withColumn(
        "codigo_concepto",
        F.concat(
            F.lit("SFC_FILA_"),
            F.lpad(
                F.col("fila_excel").cast("string"),
                4,
                "0"
            )
        )
    )

    # Identificador determinístico del indicador
    .withColumn(
        "id_indicador",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("archivo_origen"),
                F.col("columna_excel").cast("string"),
                F.col("codigo_concepto"),
                F.col("fecha_corte").cast("string")
            ),
            256
        )
    )

    .withColumn(
        "unidad_medida",
        F.lit("POR_VALIDAR")
    )

    .withColumn(
        "fecha_transformacion",
        F.current_timestamp()
    )
)

In [ ]:
display(
    df_silver_preparado
    .groupBy(
        "anio",
        "mes",
        "fecha_corte"
    )
    .agg(
        F.count("*").alias("total_indicadores"),
        F.countDistinct("id_indicador").alias("ids_unicos"),
        F.countDistinct("codigo_concepto").alias("conceptos")
    )
    .orderBy("anio", "mes")
)

In [ ]:
from pyspark.sql import functions as F

df_silver_final = (
    df_silver_preparado
    .withColumn(
        "valor_indicador",
        F.expr("try_cast(valor_limpio AS DECIMAL(38,18))")
    )
    .withColumn(
        "estado_validacion",
        F.when(
            F.col("valor_indicador").isNotNull(),
            "VALIDO"
        )
        .when(
            F.upper(F.trim(F.col("valor_limpio"))) == "#DIV/0!",
            "ERROR_ORIGEN"
        )
        .when(
            F.col("valor_limpio").isNull() |
            (F.trim(F.col("valor_limpio")) == "") |
            F.upper(F.trim(F.col("valor_limpio"))).isin("-", "N/A", "NA"),
            "SIN_DATO"
        )
        .otherwise("ERROR_CONVERSION")
    )
    .select(
        "id_indicador",
        "codigo_entidad",
        "nombre_entidad",
        "codigo_concepto",
        "nombre_concepto",
        "fecha_corte",
        "anio",
        "mes",
        "valor_indicador",
        "unidad_medida",
        "archivo_origen",
        "fecha_ingesta",
        "fecha_transformacion",
        "estado_validacion"
    )
)

In [ ]:
display(
    df_silver_final
    .groupBy("anio", "estado_validacion")
    .agg(
        F.count("*").alias("total_registros"),
        F.count("valor_indicador").alias("valores_numericos")
    )
    .orderBy("anio", "estado_validacion")
)

In [ ]:
from pyspark.sql import functions as F

df_integridad = df_silver_final.agg(
    F.count("*").alias("total_registros"),

    F.countDistinct("id_indicador").alias("ids_unicos"),

    F.sum(
        F.when(F.col("codigo_entidad").isNull(), 1)
         .otherwise(0)
    ).alias("entidades_nulas"),

    F.sum(
        F.when(F.col("codigo_concepto").isNull(), 1)
         .otherwise(0)
    ).alias("conceptos_nulos"),

    F.sum(
        F.when(F.col("fecha_corte").isNull(), 1)
         .otherwise(0)
    ).alias("fechas_nulas"),

    F.sum(
        F.when(
            (F.col("estado_validacion") == "VALIDO") &
            F.col("valor_indicador").isNull(),
            1
        ).otherwise(0)
    ).alias("validos_sin_valor")
)

display(df_integridad)

In [ ]:
tabla_destino = "catalog_au.silver.sfc_indicadores_v2"

schema_transform = [
    (f.name, f.dataType.simpleString())
    for f in df_silver_final.schema.fields
]

schema_destino = [
    (f.name, f.dataType.simpleString())
    for f in spark.table(tabla_destino).schema.fields
]

print("Esquema Transform:")
for campo in schema_transform:
    print(campo)

print("\nEsquemas compatibles:", schema_transform == schema_destino)